<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/ViT-Experiments/ViT_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install -q transformers torch

In [24]:
!pip install -q timm

In [25]:
!pip install -q evaluate

In [26]:
import timm
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Using a smaller ViT model for reduced RAM usage
model_id = "vit_large_patch16_rope_mixed_ape_224"

model = timm.create_model(model_id, pretrained=True)
model.eval()

config = timm.data.resolve_model_data_config(model)
print(config)
vit_transform = timm.data.create_transform(**config)
print(vit_transform)

# Load an AutoModelForImageClassification from Hugging Face Transformers to get ImageNet-1k id2label mapping
# This will be used for both ViT and ResNet predictions for consistency.
id2label_model = AutoModelForImageClassification.from_pretrained("google/vit-base-patch16-224")
id2label_mapping = id2label_model.config.id2label

print(f"Loaded ViT model: {model_id}")

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.9, 'crop_mode': 'center'}
Compose(
    Resize(size=248, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loaded ViT model: vit_large_patch16_rope_mixed_ape_224


## Compare ViT vs ResNet behavior

To compare the Vision Transformer (ViT) with a Convolutional Neural Network (CNN) like ResNet, we'll load a ResNet model and classify the same image. We'll use the `timm` library for this, which provides access to a wide range of pre-trained models.

In [27]:
import timm
import torch
import torchvision.transforms as transforms

# Load a smaller ResNet model (resnet18) for reduced RAM usage
resnet_model_id = "resnet101"
resnet_model = timm.create_model(resnet_model_id, pretrained=True)
resnet_model.eval()

# Get the preprocessing transforms for the ResNet model
# These transforms are typically the same across most ImageNet models (resize, center crop, normalize)
config = timm.data.resolve_model_data_config(resnet_model)
resnet_transform = timm.data.create_transform(**config)
print(resnet_transform)

print(f"Loaded ResNet model: {resnet_model_id}")

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

Compose(
    Resize(size=235, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)
Loaded ResNet model: resnet101


In [28]:
from datasets import load_dataset

food = load_dataset("ethz/food101", split="train[:5000]")
food = food.train_test_split(test_size=0.2)

In [29]:
labels = food["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [30]:
from transformers import AutoImageProcessor

checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [31]:
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])

In [32]:
def transforms(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples

food = food.with_transform(transforms)

In [33]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [34]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Freezing Backbone Layers

To fine-tune only the classification head, we need to freeze the parameters of the model's feature extractor (the backbone). This means we'll set `requires_grad=False` for all parameters except those belonging to the `model.head` module.

In [35]:
print(model)

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classification head (typically named 'head' in timm models)
# You might need to inspect the model architecture (e.g., print(model)) if it's named differently
for param in model.classifier.parameters():
    param.requires_grad = True

print("Model parameters frozen. Only classification head will be fine-tuned.")

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [36]:
training_args = TrainingArguments(
    output_dir="my_awesome_food_model",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss


NameError: name 'accuracy' is not defined

## Fine-tuning the Whole Model (All Layers Unfrozen)

As requested, this section will fine-tune the entire Vision Transformer model (all layers, not just the classification head) on the Food101 dataset for 10 epochs.

In [ ]:
# Re-initialize the model to ensure no layers are frozen from previous steps
model_full_finetune = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Explicitly ensure all parameters require gradients for full fine-tuning
for param in model_full_finetune.parameters():
    param.requires_grad = True

print("Model re-initialized. All parameters are unfrozen and will be trained.")
print(model_full_finetune)

In [ ]:
training_args_full = TrainingArguments(
    output_dir="my_awesome_food_model_full_finetune", # New output directory
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_full = Trainer(
    model=model_full_finetune,
    args=training_args_full,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting full model fine-tuning...")
trainer_full.train()
print("Full model fine-tuning complete.")

## LoRA Fine-tuning

This section demonstrates fine-tuning the Vision Transformer model using Low-Rank Adaptation (LoRA) to reduce computational cost and memory footprint during training.

In [ ]:
!pip install -q peft
!pip install --upgrade -q torchao

In [ ]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config_dict = {
    "r": 16,  # LoRA attention dimension
    "lora_alpha": 32,  # Alpha parameter for LoRA scaling
    "target_modules": ["query", "value"], # Target modules for LoRA. For ViT, 'query' and 'value' are common.
    "lora_dropout": 0.1,  # Dropout probability for LoRA layers
    "bias": "none",  # Bias type for LoRA layers
    "task_type": "CAUSAL_LM" # Task type. Set to 'CAUSAL_LM' for now, will adjust if needed.
}

# Re-initialize the base model
model_lora_base = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Adjust task type if necessary for image classification
# The task type 'CAUSAL_LM' is often used for text, for image classification, a different task_type might be more appropriate
# For image classification, there isn't a direct PEFT task_type, but we'll adapt.
# Let's set it to 'SEQ_CLS' (sequence classification) which is a common fallback for classification tasks if no specific image task type exists.
lora_config_dict['task_type'] = 'SEQ_CLS'
lora_config = LoraConfig(**lora_config_dict)

# Wrap the base model with LoRA
model_lora = get_peft_model(model_lora_base, lora_config)

print("LoRA model created:")
model_lora.print_trainable_parameters()
print(model_lora)

In [ ]:
# Set up TrainingArguments for LoRA fine-tuning
training_args_lora = TrainingArguments(
    output_dir="my_awesome_food_model_lora", # New output directory for LoRA
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5, # Start with a few epochs for LoRA
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_lora_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
trainer_lora.train()
print("LoRA fine-tuning complete.")

# Task
The goal is to compare different fine-tuning strategies (head-only, full model, and LoRA) for a Vision Transformer (ViT) model on the Food101 dataset. The comparison will be based on performance metrics (accuracy and F1-score) and the number of trainable parameters. Additionally, the models will be evaluated on a separate CIFAR-10 dataset to assess their generalization capabilities.

## Install Additional Libraries

### Subtask:
Install necessary libraries such as `scikit-learn` for F1 score and confusion matrix, `matplotlib` and `seaborn` for plotting, and `accelerate` for optimized training.


**Reasoning**:
I need to install `scikit-learn` to calculate F1 score and confusion matrix.



In [ ]:
!pip install -q scikit-learn

**Reasoning**:
Now I will install the `matplotlib`, `seaborn`, and `accelerate` libraries.



In [ ]:
!pip install -q matplotlib seaborn accelerate

## Define Parameter Counting Utility

### Subtask:
Create a Python function to count and display the number of trainable parameters in a given PyTorch model. This function will be reused at various stages to track model complexity.


**Reasoning**:
The subtask requires defining a function to count trainable parameters. This code block implements that function by iterating through model parameters and summing up the elements of those that require gradients, then printing the result in millions.



In [ ]:
def count_parameters(model):
    """
    Counts and displays the number of trainable parameters in a PyTorch model.
    """
    num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Number of trainable parameters: {num_params_trainable:,} ({num_params_trainable / 1e6:.2f} million)")
    print(f"Total parameters: {total_params:,} ({total_params / 1e6:.2f} million)")


## Remove Single Image Processing

### Subtask:
Remove the existing cells that prepare and classify a single image, as the evaluation will now focus on test sets.


**Reasoning**:
Deleting the markdown cell that introduces loading a single image.



## Prepare Food101 Dataset

### Subtask:
Load the 'ethz/food101' dataset, create training and testing splits, and apply the defined image transformations. This will be the primary dataset for fine-tuning and evaluation.


**Reasoning**:
Confirming the transformations and displaying the dataset structure to ensure all previous steps for dataset preparation were successful.



In [ ]:
print("Sample from transformed dataset (first item of train split):\n", food["train"][0])
print("\nFood dataset structure:\n", food)


## Prepare CIFAR-10 Dataset

### Subtask:
Load the CIFAR-10 dataset, create training and testing splits, and apply necessary image transformations. This dataset will be used to evaluate the model's generalization capabilities.

In [ ]:
from datasets import load_dataset

cifar10 = load_dataset('uoft-cs/cifar10')

# CIFAR-10 typically has 'train' and 'test' splits by default
# No need for explicit train_test_split unless different proportions are desired
print(f"CIFAR-10 dataset structure: {cifar10}")

The CIFAR-10 dataset contains images of size 32x32. We need to define new image transformations suitable for these smaller images, or ensure our existing transformations can handle them correctly. For consistency with the ViT model, we will resize them to 224x224.

In [ ]:
from torchvision.transforms import Resize, Compose, Normalize, ToTensor
from transformers import AutoImageProcessor # Re-import AutoImageProcessor

# CIFAR-10 images are 32x32, but our ViT model expects 224x224.
# We'll use the same normalization parameters but adjust the resizing.

# The existing image_processor from the ViT model was for 224x224 input.
# We can reuse its mean and std, but adjust the initial resize.

# Re-initialize image_processor here to ensure it's defined
checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)

normalize_cifar = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

# For CIFAR-10, we'll resize to 224x224 directly for consistency with ViT model input
cifar_transforms = Compose([
    Resize((224, 224)), # Resize to ViT expected input size
    ToTensor(),
    normalize_cifar
])

def transforms_cifar(examples):
    examples['pixel_values'] = [cifar_transforms(img.convert('RGB')) for img in examples['img']]
    del examples['img'] # CIFAR-10 uses 'img' as the key for images
    return examples

cifar10_processed = cifar10.with_transform(transforms_cifar)

# Update CIFAR-10 label mappings
cifar_labels = cifar10['train'].features['label'].names
cifar_id2label, cifar_label2id = dict(), dict()
for i, label in enumerate(cifar_labels):
    cifar_label2id[label] = str(i)
    cifar_id2label[str(i)] = label

print("Sample from transformed CIFAR-10 dataset (first item of train split):\n", cifar10_processed['train'][0])

## Update `compute_metrics` Function

### Subtask:
Modify the `compute_metrics` function to include the F1-score in addition to accuracy. This will provide a more comprehensive evaluation metric, especially for imbalanced datasets.

In [ ]:
import evaluate
import numpy as np

# Load the F1 metric
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    # Compute accuracy
    accuracy_result = accuracy.compute(predictions=predictions, references=labels)

    # Compute F1-score. Use 'weighted' average for multi-class classification.
    f1_result = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    # Combine results
    return {
        "accuracy": accuracy_result["accuracy"],
        "f1": f1_result["f1"]
    }

## Define Reusable Evaluation and Plotting Function

### Subtask:
Create a function that takes a trained model, a dataset, and label mappings as input, computes predictions, calculates accuracy and F1-score, and generates a confusion matrix. This function will be used to evaluate all fine-tuned models on both Food101 and CIFAR-10 datasets.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
from tqdm.auto import tqdm

def evaluate_and_plot(model, trainer, dataset, id2label_mapping, dataset_name, num_labels, device='cuda'):
    print(f"\n--- Evaluating on {dataset_name} --- ")

    # Make predictions
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    labels = predictions.label_ids

    # Get predicted labels
    predicted_labels = np.argmax(logits, axis=1)

    # Compute metrics using the shared compute_metrics function
    metrics = compute_metrics(predictions)
    print(f"Accuracy on {dataset_name}: {metrics['accuracy']:.4f}")
    print(f"F1-score (weighted) on {dataset_name}: {metrics['f1']:.4f}")

    # Generate Confusion Matrix
    cm = confusion_matrix(labels, predicted_labels)

    # Plot Confusion Matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues',
                xticklabels=list(id2label_mapping.values()),
                yticklabels=list(id2label_mapping.values()))
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.title(f'Confusion Matrix for {model.config._name_or_path.split("/")[-1]} on {dataset_name}')
    plt.show()

    return metrics

## Evaluate Head-Only Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the head-only fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Head-Only Fine-tuning:")
count_parameters(model)

# Evaluate the head-only fine-tuned model on Food101 test set
head_only_metrics_food101 = evaluate_and_plot(
    model=model,
    trainer=trainer,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Head-Only)",
    num_labels=len(id2label)
)
